In [2]:
import numpy as np
import pandas as pd

# Initial Inspection of Data

In [6]:
gold_df = pd.read_csv("gold_backtest_1y.csv", parse_dates=["signal_date", "entry_date"])
gold_df

,signal_date,entry_date,asset,headline_summary,sentiment_score,signal,entry_price,price_1d,price_3d,price_7d
0,2025-08-11,2025-08-12,GC=F,Tariffs As A Geopolitical Tool: Short And Long...,0.200000,⚪ HOLD,3356.199951,3358.699951,3336.000000,3336.899902
1,2025-08-12,2025-08-13,GC=F,ETF Investors Face Crosswinds: China Hits Chip...,-0.400000,🔴 SELL,3358.800049,3335.199951,3331.699951,3374.399902
2,2025-08-13,2025-08-14,GC=F,Top 10 Trending Stocks On WallStreetBets As Of...,0.000000,⚪ HOLD,3346.800049,3336.000000,3313.399902,3373.800049
3,2025-08-14,2025-08-15,GC=F,Gold Forecast: A Really Big Upleg Should Start...,1.000000,⚪ HOLD,3346.800049,3331.699951,3343.399902,3388.600098
4,2025-08-15,2025-08-18,GC=F,China's New Physical Gold Mandate For Insuranc...,0.500000,⚪ HOLD,3333.500000,3313.399902,3336.899902,3404.600098
...,...,...,...,...,...,...,...,...,...,...
238,2026-07-23,2026-07-24,GC=F,What Is Going On With Gold? | ‘Big Short’ Bill...,-0.500000,🔴 SELL,4067.600098,4074.500000,4034.699951,NaN
239,2026-07-24,2026-07-27,GC=F,The One Bond Chart That Keeps Me Up At Night |...,0.000000,⚪ HOLD,4090.100098,4036.300049,4100.100098,NaN
240,2026-07-27,2026-07-28,GC=F,Real Assets Are Beating Tech Right Now. Here's...,-0.333333,🔴 SELL,4025.699951,4034.699951,NaN,NaN
241,2026-07-28,2026-07-29,GC=F,Commodities: Oil Slides As Hopes Grow For A U....,-1.000000,🔴 SELL,4018.100098,4100.100098,NaN,NaN


In [7]:
btc_df = pd.read_csv("btc_backtest_1y.csv", parse_dates=["signal_date", "entry_date"])
btc_df

,signal_date,entry_date,asset,headline_summary,sentiment_score,signal,entry_price,price_1d,price_3d,price_7d
0,2025-08-09,2025-08-10,BTC-USD,Weekly Commentary: Anything But Normal Times,0.0,⚪ HOLD,116497.718750,118731.445312,123344.062500,117453.062500
1,2025-08-11,2025-08-12,BTC-USD,"August 2025 Macro Outlook: Fiscal Flows, Bank ...",0.0,⚪ HOLD,118717.664062,123344.062500,117398.351562,112831.179688
2,2025-08-12,2025-08-13,BTC-USD,Keith Fitz-Gerald And David Keller On Current ...,0.0,⚪ HOLD,120168.976562,118359.578125,117491.351562,114274.742188
3,2025-08-14,2025-08-15,BTC-USD,Bitcoin Hits Record High: ETFs in Focus,1.0,🟢 BUY,118365.781250,117491.351562,116252.312500,116874.085938
4,2025-08-15,2025-08-16,BTC-USD,"Markets Weekly Outlook: Jackson Hole, NZ Rate ...",0.5,🟢 BUY,117398.421875,117453.062500,112831.179688,115374.328125
...,...,...,...,...,...,...,...,...,...,...
219,2026-07-25,2026-07-26,BTC-USD,Weekly Commentary: Bond Yield Breakout,0.0,⚪ HOLD,64311.785156,63724.898438,63908.167969,NaN
220,2026-07-26,2026-07-27,BTC-USD,Savannah Guthrie Posts New Plea To Mom’s Capto...,-0.2,⚪ HOLD,65340.589844,63871.363281,64725.308594,NaN
221,2026-07-27,2026-07-28,BTC-USD,Crypto hedge fund manager gets 37 months in pr...,0.2,⚪ HOLD,63723.488281,63908.167969,NaN,NaN
222,2026-07-28,2026-07-29,BTC-USD,"FBTC: Self-Custody Was The Pitch, Now There's ...",0.0,⚪ HOLD,63871.394531,64725.308594,NaN,NaN


In [8]:
print(gold_df.info())
print(btc_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   signal_date       243 non-null    datetime64[ns]
 1   entry_date        243 non-null    datetime64[ns]
 2   asset             243 non-null    object        
 3   headline_summary  243 non-null    object        
 4   sentiment_score   243 non-null    float64       
 5   signal            243 non-null    object        
 6   entry_price       243 non-null    float64       
 7   price_1d          242 non-null    float64       
 8   price_3d          240 non-null    float64       
 9   price_7d          236 non-null    float64       
dtypes: datetime64[ns](2), float64(5), object(3)
memory usage: 19.1+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------

# Preprocessing

### Backtest Evaluation

This section contains the logic to evaluate the trading signals generated by the SignalEngine. It focuses on assessing whether predicted price movements actually occurred and calculating profitability metrics.

Only 'BUY' and 'SELL' signals are considered for evaluation; 'HOLD' signals are excluded as they represent no active position or directional prediction.

In [12]:
HORIZONS = ["price_1d", "price_3d", "price_7d"]

def _direction(signal: str) -> int:
    """Converts a signal string (e.g., '🟢 BUY') to a numerical direction (1, -1, or 0)."""
    if "BUY" in signal:
        return 1
    if "SELL" in signal:
        return -1
    return 0

def evaluate_horizon(df: pd.DataFrame, horizon_col: str) -> dict:
    """Evaluates trading performance for a given horizon column."""
    trades = df[df["signal"].apply(_direction) != 0].copy()
    trades = trades.dropna(subset=[horizon_col, "entry_price"])
    if trades.empty:
        return {"horizon": horizon_col, "n_trades": 0}

    direction = trades["signal"].apply(_direction)
    trades["return"] = direction * (trades[horizon_col] - trades["entry_price"]) / trades["entry_price"]

    gains = trades.loc[trades["return"] > 0, "return"].sum()
    losses = trades.loc[trades["return"] < 0, "return"].sum()
    profit_factor = gains / abs(losses) if losses != 0 else np.inf

    n = len(trades)
    accuracy = (trades["return"] > 0).mean()
    # 95% CI on accuracy (normal approximation) -- if this interval contains
    # 0.5, the result isn't distinguishable from a coin flip at this sample size.
    se = np.sqrt(accuracy * (1 - accuracy) / n) if n > 0 else 0
    acc_ci_low, acc_ci_high = accuracy - 1.96 * se, accuracy + 1.96 * se

    cumulative_return = (1 + trades["return"]).prod() - 1

    return {
        "horizon": horizon_col,
        "n_trades": n,
        "signal_accuracy": accuracy,
        "accuracy_95ci": f"[{acc_ci_low:.2f}, {acc_ci_high:.2f}]",
        "avg_return_per_trade": trades["return"].mean(),
        "profit_factor": profit_factor,
        "cumulative_return": cumulative_return,
    }

def buy_and_hold_return(df: pd.DataFrame, horizon_col: str) -> float:
    """Calculates buy and hold return for the same time window as the strategy's trades."""
    valid = df.dropna(subset=[horizon_col, "entry_price"])
    if valid.empty:
        return np.nan
    start_price = valid["entry_price"].iloc[0]
    end_price = valid[horizon_col].iloc[-1]
    return (end_price - start_price) / start_price

def run_report(df: pd.DataFrame, asset_label: str) -> pd.DataFrame:
    """Runs a full backtest report for a given DataFrame and asset label."""
    rows = [evaluate_horizon(df, h) for h in HORIZONS]
    report = pd.DataFrame(rows)
    report["buy_and_hold_return"] = [buy_and_hold_return(df, h) for h in HORIZONS]
    report.insert(0, "asset", asset_label)
    return report

# Set display options for better DataFrame visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

### Generate Gold Backtest Report

In [13]:
gold_report = run_report(gold_df, "Gold")
display(gold_report)

,asset,horizon,n_trades,signal_accuracy,accuracy_95ci,avg_return_per_trade,profit_factor,cumulative_return,buy_and_hold_return
0,Gold,price_1d,98,0.500000,"[0.40, 0.60]",0.003449,1.520555,0.356101,0.22165
1,Gold,price_3d,96,0.583333,"[0.48, 0.68]",0.004176,1.416032,0.419960,0.22165
2,Gold,price_7d,95,0.536842,"[0.44, 0.64]",0.001852,1.101771,0.065005,0.22165


### Generate Bitcoin Backtest Report

In [14]:
btc_report = run_report(btc_df, "Bitcoin")
display(btc_report)

,asset,horizon,n_trades,signal_accuracy,accuracy_95ci,avg_return_per_trade,profit_factor,cumulative_return,buy_and_hold_return
0,Bitcoin,price_1d,84,0.535714,"[0.43, 0.64]",0.001141,1.095391,0.041373,-0.444407
1,Bitcoin,price_3d,84,0.464286,"[0.36, 0.57]",0.000418,1.026720,-0.036458,-0.444407
2,Bitcoin,price_7d,83,0.493976,"[0.39, 0.60]",0.007354,1.364943,0.572040,-0.444407


We notice BTC has a harsh **Buy&Hold** return which might be due to the hard crash the market had at the time

### Combine and Display Full Report

In [ ]:
full_report = pd.concat([gold_report, btc_report], ignore_index=True)
display(full_report)